Extract frames with OpenCV

In [ ]:
import cv2
import math

video_path = r"C://Users//Avi//Desktop//Projectwork//Intelligent-Exploration-of-Educational-Videos//initial_test//STREAM_PROJECT_FAIRDI-1105FCDE.mp4"
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval_sec = 10  # one frame every 2 seconds
frame_interval = int(fps * frame_interval_sec)

frames = []  # list of (timestamp_sec, frame_image)

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % frame_interval == 0:
        ts = frame_idx / fps
        frames.append((ts, frame))
    frame_idx += 1

cap.release()


In [25]:
import os
output_dir = r"C://Users//Avi//Desktop//Projectwork//Intelligent-Exploration-of-Educational-Videos//initial_test//frames"
os.makedirs(output_dir, exist_ok=True)

for i, (ts, frame) in enumerate(frames):
    filename = os.path.join(output_dir, f"frame_{i:04d}_ts_{ts:.1f}.jpg")
    cv2.imwrite(filename, frame)

In [26]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
import torch
import json

# Load model (downloads ~2GB first run)
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16)
model.eval()

def caption_frames(frames, output_json="captions.json"):
    captions = []
    with torch.no_grad():
        for ts, frame in frames:
            # Convert BGR (cv2) to RGB and PIL Image
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)
            
            inputs = processor(pil_image, return_tensors="pt")
            generated_ids = model.generate(**inputs, max_length=50)
            caption = processor.decode(generated_ids[0], skip_special_tokens=True)
            
            captions.append({"timestamp": ts, "caption": caption})
    
    with open(output_json, 'w') as f:
        json.dump(captions, f, indent=2)
    print(f"Saved {len(captions)} captions to {output_json}")

# After your frame extraction loop:
caption_frames(frames)


Loading checkpoint shards: 100%|██████████| 2/2 [00:21<00:00, 10.78s/it]


Saved 29 captions to captions.json


In [22]:
from transformers import BlipImageProcessor, BlipForConditionalGeneration
from PIL import Image
import torch
import cv2

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Salesforce/blip-image-captioning-base"

processor = BlipImageProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name).to(device)

def caption_frame(frame_bgr):
    img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)
    
    inputs = processor(pil_img, return_tensors="pt").to(device)
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_length=50, num_beams=5)
    
    # Use processor's tokenizer
    generated_text = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return generated_text


In [23]:
test_frame = frames[0][1]
print(caption_frame(test_frame))

AttributeError: 'BlipImageProcessor' object has no attribute 'tokenizer'

Loop over frames:

In [20]:
captions = []
for ts, frame in frames[:10]:  # test 10 frames first
    print(f"Frame at {ts:.1f}s: {caption_frame(frame)}")
    captions.append({"time": ts, "caption": caption_frame(frame)})


Frame at 0.0s: a graphic of a text and data
Frame at 10.0s: a graphic of a data and data
Frame at 20.0s: a graphic of a data and data
Frame at 30.0s: a graphic of a data and data
Frame at 40.0s: a line of data and data
Frame at 50.0s: our data and data are connected by data and data.
Frame at 60.0s: our task is showing the structure of data.
Frame at 70.0s: our task is showing the structure of data.
Frame at 80.0s: data on the data and data
Frame at 90.0s: our task is to create a structure of data.


Save captions

In [21]:
import json

with open("frame_captions.json", "w", encoding="utf-8") as f:
    json.dump(captions, f, ensure_ascii=False, indent=2)
